In [1]:
import torch
import torch.nn.functional as F
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [2]:
from tqdm import tqdm
import gymnasium as gym
import gym_donkeycar
import numpy as np
conf = {
    "host": "127.0.0.1",         # 模擬器主機
    "port": 9091,                # 模擬器連接埠
    "log_level": 20,             # logging.INFO
    "level": "generated_track",  # 場景名稱
    "cam_resolution": (120, 160, 3),  # camera resolution (H,W,D)
    "max_cte": 2.0,              # max cross track error
    "body_style": "donkey",
    "car_name":"kghs",
    "body_rgb": (255, 255, 255),
    "font_size": 20
}

In [3]:
class ImageEnv(gym.Wrapper):
    def __init__(self,env,stack_frames=4,delay_op=5):
        super(ImageEnv,self).__init__(env)
        self.stack_frames=stack_frames
        self.delay_op=delay_op
        steer_levels=np.linspace(-1.0,1.0,20,dtype=np.float32)
        strottle_levels=np.linspace(0.0,1.0,20,dtype=np.float32)
        self._action_table=np.array([(s,t) for s in steer_levels for t in strottle_levels],dtype=np.float32)
        self.action_space=gym.spaces.Discrete(self._action_table.shape[0])
    def reset(self):
        s,info=self.env.reset()
        for i in range(self.delay_op):
            s,r,done,stop,info=self.env.step(np.array([0.0,0.0]))
            img=s
            s=(s.dot([0.299,0.587,0.114])/255)-0.5
            self.stacked_state=np.tile( s , (self.stack_frames,1,1) )#(4,120,160)
        return self.stacked_state,img
    def step(self,action):
        reward=0
        for _ in range(self.stack_frames):
            s,r,done,stop,info=self.env.step(self._action_table[action])
            img=s
            s=(s.dot([0.299,0.587,0.114])/255)-0.5 
            reward+=r
            if done or stop:break
            self.stacked_state=np.concatenate( (self.stacked_state[1:] , s[np.newaxis] ) , axis=0 )
        return self.stacked_state,reward,done,stop,img

In [4]:
class ReplyBuffer():
    def __init__(self,max_size=10000,num_step=1):
        self.s=torch.zeros((max_size,4,120,160)).to(device)
        self.a=torch.zeros((max_size)).to(device)
        self.r=torch.zeros((max_size,1)).to(device)
        self.s_=torch.zeros((max_size,4,120,160)).to(device)
        self.done=torch.zeros((max_size,1)).to(device)
        self.ptr=0
        self.size=0
        self.max_size=max_size
        self.num_step=num_step
    def append(self,s,a,r,s_,done):
        self.s[self.ptr]=torch.tensor(s)
        self.s_[self.ptr]=torch.tensor(s_)
        self.a[self.ptr]=a
        self.r[self.ptr]=r
        self.done[self.ptr]=done
        self.ptr=(self.ptr+1)%self.max_size
        self.size=min(self.size+1,self.max_size)
    def sample(self,batch_size):
        ind = torch.randint(0, self.size, (batch_size,))
        return self.s[ind],self.a[ind],self.r[ind],self.s_[ind],self.done[ind]        

In [5]:
class DQN(torch.nn.Module):
    def __init__(self,n_act):
        super(DQN,self).__init__()
        self.conv1=torch.nn.Conv2d(4,16,8,4)#輸入色頻、輸出色頻、卷積核大小、步長=>16x29x39
        self.conv2=torch.nn.Conv2d(16,32,4,2)#輸入色頻、輸出色頻、卷積核大小、步長=>32x13x18
        self.fc1=torch.nn.Linear(7488,512)
        self.fc2=torch.nn.Linear(512,n_act)
    def forward(self,x):
        x=F.relu( self.conv1(x) )
        x=F.relu( self.conv2(x) )
        x=x.reshape((-1,7488))
        x=F.relu(self.fc1(x))
        x=self.fc2(x)
        return x

In [6]:
env = gym.make("donkey-generated-track-v0", conf=conf,render_mode="rgb_array")
env = ImageEnv(env)

INFO:gym_donkeycar.core.client:connecting to 127.0.0.1:9091 
C:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
C:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


starting DonkeyGym env
Setting default: start_delay 5.0
Setting default: frame_skip 1
Setting default: steer_limit 1.0
Setting default: throttle_min 0.0
Setting default: throttle_max 1.0
loading scene generated_track


INFO:gym_donkeycar.envs.donkey_sim:on need car config
INFO:gym_donkeycar.envs.donkey_sim:sending car config.
INFO:gym_donkeycar.envs.donkey_sim:sim started!


In [10]:
Log={"TrainReward":[],"TestReward":[],"Loss":[]}
old_i=0
old_dqn=f"DQN_{old_i}.pt"

In [11]:
class DQNAgent():
    def __init__(self,gamma=0.9,eps_low=0.1,lr=0.00001):
        self.env=env
        self.n_act=self.env.action_space.n
        self.PredictDQN=DQN(self.n_act)
        self.TargetDQN=DQN(self.n_act)
        if old_i>0:
            self.PredictDQN.load_state_dict(torch.load(old_dqn))
            self.TargetDQN.load_state_dict(torch.load(old_dqn))
        self.PredictDQN.to(device)
        self.TargetDQN.to(device)
        self.LossFun=torch.nn.SmoothL1Loss()
        self.optimizer=torch.optim.Adam( self.PredictDQN.parameters() , lr=lr )
        self.gamma=gamma
        self.eps_low=eps_low
        self.rb=ReplyBuffer(max_size=10000,num_step=1)
    def PredictA(self,s):
        with torch.no_grad():
            return torch.argmax(self.PredictDQN(torch.FloatTensor(s).to(device))).item()
    def SelectA(self,a):
        return self.env.action_space.sample() if np.random.random()<self.EPS else a 
    def Test(self):
        total_reward=0
        s,_=self.env.reset()
        while True:
            a=self.PredictA(s)
            s,r,done,stop,_=self.env.step(a)
            total_reward+=r
            if done or stop : break
        Log["TestReward"].append(total_reward)
    def Train(self,N_EPISODES):
        for i in tqdm(range(N_EPISODES)):
            self.EPS=1-( i * (1-self.eps_low) / N_EPISODES )
            total_reward=0
            s,_=self.env.reset()
            while True:
                a=self.SelectA(self.PredictA(s))
                s_,r,done,stop,_=self.env.step(a)
                self.rb.append(s,a,r,s_,done)
                if  self.rb.size > 256 and i % self.rb.num_step==0:
                    self.learn()
                total_reward+=r
                s=s_
                if done or stop : break
            print(i,total_reward)
            Log["TrainReward"].append(total_reward)
            if i % 20==19 : self.TargetDQN.load_state_dict(self.PredictDQN.state_dict())
            if i % 500==499:
                torch.save(self.PredictDQN.state_dict(),f"DQN_{i}.pt")
                self.Test()
    def learn(self):
        self.optimizer.zero_grad()
        b_s,b_a,b_r,b_s_,b_done=self.rb.sample(256)
        PredictQ=(self.PredictDQN(b_s)*F.one_hot(b_a.long(),self.n_act)).sum(1,keepdims=True)
        TargetQ=b_r + (1-b_done)*self.gamma*self.TargetDQN(b_s_).max(1,keepdims=True)[0]
        loss=self.LossFun(PredictQ,TargetQ)
        Log["Loss"].append(float(loss))
        loss.backward()
        self.optimizer.step()

In [13]:
Agent=DQNAgent(gamma=0.9,eps_low=0.1,lr=0.0005)
Agent.Train(N_EPISODES=20000)

  0%|                                                                             | 1/20000 [00:02<13:07:28,  2.36s/it]

0 8.411124901581424


  0%|                                                                             | 2/20000 [00:06<18:09:32,  3.27s/it]

1 43.01933766747249


  0%|                                                                             | 2/20000 [00:07<20:27:52,  3.68s/it]


KeyboardInterrupt: 

In [14]:
Agent.Test()